### Importación de librerías necesárias

In [4]:
import unicodedata
import pandas as pd
import numpy as np
import re

In [ ]:
from src.preprocessing import build_model_text

### Importando el dataset

In [5]:
df_en = pd.read_csv('../Data/raw_data_en.csv')

In [6]:
print(df_en.shape)
df_en.head()

(6000, 5)


,title,text,subject,date,label
0,WATCH: Trump Just Told All The Anti-Gay Bigot...,A whole lot of evangelical Trump voters just d...,News,"November 13, 2016",0
1,"China backs U.N. call for justice in Yemen, U....",GENEVA (Reuters) - China signaled on Wednesday...,worldnews,"September 13, 2017",1
2,THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...,,politics,"Feb 16, 2017",0
3,WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...,THE WSJ S MARY KISSEL NAILS IT ON THE DEM DEBA...,politics,"Nov 15, 2015",0
4,Lawmakers aim to delay U.S. ceding control of ...,WASHINGTON (Reuters) - Critics of a plan for t...,politicsNews,"September 13, 2016",1


In [7]:
df_clean = df_en.copy()

### Drop empty `text`:

In [8]:
empty_text_mask = (
    df_clean["text"]
    .astype(str)
    .str.strip()
    .eq("")
)

print("Empty texts before:", empty_text_mask.sum())

df_clean = df_clean.loc[
    ~empty_text_mask
].copy()

print("Shape after removing empty texts:", df_clean.shape)

Empty texts before: 66
Shape after removing empty texts: (5934, 5)


Aqui no hacemos `df_clean.dropna()` porque el problema detectado no se detectaba así en la fase EDA.

### Drop Duplicates:

In [9]:
print(
    "Duplicated texts before:",
    df_clean.duplicated(subset=["text"]).sum()
)

df_clean = (
    df_clean
    .drop_duplicates(
        subset=["text"],
        keep="first"
    )
    .copy()
)

print(
    "Duplicated texts after:",
    df_clean.duplicated(subset=["text"]).sum()
)

print("Shape:", df_clean.shape)

Duplicated texts before: 106
Duplicated texts after: 0
Shape: (5828, 5)


In [10]:
duplicate_titles_after_text_dedup = (
    df_clean
    .duplicated(subset=["title"])
    .sum()
)

print("Duplicated titles after text deduplicates:",
      duplicate_titles_after_text_dedup)

Duplicated titles after text deduplicates: 8


In [11]:
pd.set_option('display.max_colwidth',1000)

In [12]:
duplicated_titles = df_clean[
    df_clean["title"].duplicated(keep=False)
].sort_values("title")


display(
    duplicated_titles[
        ["title","text","label"]
    ]
)

,title,text,label
296,China hopes all sides' words and actions reduce tension on Korean peninsula,"BEIJING (Reuters) - China said on Monday that it hopes all sides words and actions can help reduce tensions on the Korean peninsula, after Japanese Prime Minister Shinzo Abe said Japan would shoot down North Korean missiles if necessary. Foreign Ministry spokeswoman Hua Chunying made the comment at a regular news briefing.",1
2269,China hopes all sides' words and actions reduce tension on Korean peninsula,"BEIJING (Reuters) - China said on Monday that it hopes all sides’ words and actions can help reduce tensions on the Korean peninsula, after Japanese Prime Minister Shinzo Abe said Japan would shoot down North Korean missiles if necessary. Foreign Ministry spokeswoman Hua Chunying made the comment at a regular news briefing.",1
2899,China urges North Korea not to go further in a 'dangerous direction',"UNITED NATIONS (Reuters) - China s foreign minister on Thursday called on North Korea not to go further in a dangerous direction with its nuclear program and said negotiations were the only way out of the crisis over Pyongyang s weapons development. Wang Yi also told the annual U.N. General Assembly China was committed to the denuclearization of the Korean Peninsula and there should be no new nuclear weapons north or south of the border, or elsewhere in Northeast Asia. He urged the United States to honor its four no commitment, an apparent reference to an Aug. 1 statement by U.S. Secretary of State Rex Tillerson, in which he said Washington did not seek the collapse or change of the North Korean government, accelerated reunification of the peninsula, or to send its military north of the border. We urge the DPRK not to go further along a dangerous direction, Wang said, referring to North Korea by the acronym of its official name, the Democratic People s Republic of Korea. A...",1
3409,China urges North Korea not to go further in a 'dangerous direction',"UNITED NATIONS (Reuters) - China’s foreign minister on Thursday called on North Korea not to go further in a “dangerous direction” with its nuclear program and said negotiations were the only way out of the crisis over Pyongyang’s weapons development. Wang Yi also told the annual U.N. General Assembly China was committed to the denuclearization of the Korean Peninsula and there should be no new nuclear weapons north or south of the border, or elsewhere in Northeast Asia. He urged the United States to honor its “four ‘no’ commitment,” an apparent reference to an Aug. 1 statement by U.S. Secretary of State Rex Tillerson, in which he said Washington did not seek the collapse or change of the North Korean government, accelerated reunification of the peninsula, or to send its military north of the border. “We urge the DPRK not to go further along a dangerous direction,” Wang said, referring to North Korea by the acronym of its official name, the Democratic People’s Republic of Korea. “A...",1
995,Ex-president George H.W. Bush moved to intensive care; wife hospitalized,"(Reuters) - Former U.S. President George H.W. Bush has been moved to an intensive-care unit at a Houston hospital with pneumonia and was stable and resting comfortably after doctors performed a procedure to clear his airway, his office said on Wednesday. His wife of 72 years, former first lady Barbara Bush, also was admitted to the same hospital on Wednesday as a precaution after experiencing fatigue and coughing, the office said in a statement. Bush, who at 92 is the nation’s oldest living ex-president, has been at Houston Methodist Hospital since Saturday after experiencing shortness of breath, family spokesman Jim McGrath said on Wednesday. Since then, Bush experienced an “acute respiratory problem stemming from pneumonia” and was sedated for the unspecified procedure, his office said. He will remain in the hospital’s intensive-care unit for observation, his office said. Bush and the 91-year-old former first lady marked 

In [13]:
example_titles = duplicated_titles.iloc[0]["title"]

example = df_clean[
    df_clean["title"] == example_titles
][["title","text","label"]]

display(example)

,title,text,label
296,China hopes all sides' words and actions reduce tension on Korean peninsula,"BEIJING (Reuters) - China said on Monday that it hopes all sides words and actions can help reduce tensions on the Korean peninsula, after Japanese Prime Minister Shinzo Abe said Japan would shoot down North Korean missiles if necessary. Foreign Ministry spokeswoman Hua Chunying made the comment at a regular news briefing.",1
2269,China hopes all sides' words and actions reduce tension on Korean peninsula,"BEIJING (Reuters) - China said on Monday that it hopes all sides’ words and actions can help reduce tensions on the Korean peninsula, after Japanese Prime Minister Shinzo Abe said Japan would shoot down North Korean missiles if necessary. Foreign Ministry spokeswoman Hua Chunying made the comment at a regular news briefing.",1


In [14]:
texts = example["text"].tolist()

print("Same exact text", texts[0] == texts[1])
print("lenght text 1", len(texts[0]))
print("lenght text 2", len(texts[1]))

Same exact text False
lenght text 1 327
lenght text 2 327


### Check Class Distribution:

In [15]:
class_distribution = (
    df_clean["label"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)

class_distribution["percentage"] = (
    class_distribution["count"]
    / len(df_clean)
    * 100
)

display(class_distribution)

,count,percentage
label,,
0,2833,48.610158
1,2995,51.389842


### Drop variables:

subject -> leakage <br>
date    -> sesgo temporal / metadata

In [16]:
df_model = df_clean[
    ["title", "text", "label"]
].copy()

In [17]:
df_model

,title,text,label
0,WATCH: Trump Just Told All The Anti-Gay Bigots And Mike Pence To Go F*ck Themselves,"A whole lot of evangelical Trump voters just discovered they got duped.By picking Mike Pence as his vice-president, Donald Trump sent a message to the LGBT community that their rights are in jeopardy of being rolled back.In fact, one of the reasons why so many conservative Christians tossed their alleged morality out the window to vote for him was precisely because they believed he would pass a constitutional amendment banning same-sex marriage or pack the courts with anti-gay bigots to strike down all the rulings in favor of marriage equality.But during his interview on 60 minutes with Leslie Stahl on Sunday night, Trump once again backed off another campaign promise. Do you support marriage equality? Stahl asked Trump. It s irrelevant because it s already settled, Trump replied. It s law. It was settled in the Supreme Court. I mean, it s done. When Stahl pointed out that Trump s Supreme Court nominees could reverse Obergefell v. Hodges, the case that made same-sex marriage...",0
1,"China backs U.N. call for justice in Yemen, U.S. and Saudis don't","GENEVA (Reuters) - China signaled on Wednesday it was willing to back an international inquiry into atrocities in Yemen, as demanded by the U.N. High Commissioner for Human Rights, but Saudi Arabia and the United States said they did not support the idea. For three years running U.N. human rights chief Zeid Ra ad al-Hussein has asked the 47 countries in the U.N. Human Rights Council to set up an independent investigation into Yemen s war, which has killed at least 10,000, destroyed the economy, led to a cholera epidemic and pushed millions to the brink of famine. Despite his pleas, they have twice supported a Saudi plan to let Yemen investigate by itself. On Wednesday, the Netherlands and Canada unveiled a draft resolution to establish an international commission of inquiry (COI) to ensure that perpetrators of violations and abuses, including those that may constitute war crimes and crimes against humanity, are held accountable . The three-page text was supported by many count...",1
3,WSJ REPORTER RIPS INTO DEM CANDIDATES For Their Lame Rhetoric On ISIS: “Remarkable displays of unintelligible garbage” [Video],"THE WSJ S MARY KISSEL NAILS IT ON THE DEM DEBATE WITH THE THREE CANDIDATES WHO WERE GIVING NO SOLUTIONS ON ISIS: It was one of the most remarkable displays of unintelligible garbage rhetoric that I have ever seen. Amen to that!Mary Kissel: The barbarians are at the gates. We ve seen attacks now from London to Madrid to Beirut to over the Sinai. It is time for American leadership. Hillary said this isn t America s fight. Look, Democrats are dangerously divorced from reality. Hillary says that. President Obama wants to close Gitmo. He said we contained these people. No, we haven t contained them. Bernie Sanders, who s leading in some polls, says the greatest challenge is climate change. What we need is American leadership and the majority of the American people understand we need to send troops back to the Middle East before this global disorder comes to our shores.Marie Bartiromo: I thought it was actually extraordinary, the debate last night. There was no real solution from any o...",0
4,Lawmakers aim to delay U.S. ceding control of Internet's management,"WASHINGTON (Reuters) - Critics of a plan for the U.S. government to cede control of the Internet’s technical management to other countries might succeed in delaying such a move in Congress, a senior lawmaker said on Tuesday. Senator John Thune, a senior Republican from South Dakota, told reporters lawmakers are “trying to work out ... what would be effective in terms of slowing this.” The U.S. Commerce Department oversees the Internet’s management largely because it was invented in the United States. Some Republican politicians, including Senator Ted Cruz of Texas, want to block the handover to global stakeholders, such 

### Preprocessing:

In [18]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    return text


In [19]:
def build_model_text(title, text):
    title = normalize_text(title)
    text = normalize_text(text)

    if not text:
        raise ValueError("Article body is required.")

    if title:
        return f"{title}. {text}"

    return text

In [20]:
df_model["content"] = df_model.apply(
    lambda row: build_model_text(
        row["title"],
        row["text"]
    ),
    axis=1
)

In [21]:
df_model

,title,text,label,content
0,WATCH: Trump Just Told All The Anti-Gay Bigots And Mike Pence To Go F*ck Themselves,"A whole lot of evangelical Trump voters just discovered they got duped.By picking Mike Pence as his vice-president, Donald Trump sent a message to the LGBT community that their rights are in jeopardy of being rolled back.In fact, one of the reasons why so many conservative Christians tossed their alleged morality out the window to vote for him was precisely because they believed he would pass a constitutional amendment banning same-sex marriage or pack the courts with anti-gay bigots to strike down all the rulings in favor of marriage equality.But during his interview on 60 minutes with Leslie Stahl on Sunday night, Trump once again backed off another campaign promise. Do you support marriage equality? Stahl asked Trump. It s irrelevant because it s already settled, Trump replied. It s law. It was settled in the Supreme Court. I mean, it s done. When Stahl pointed out that Trump s Supreme Court nominees could reverse Obergefell v. Hodges, the case that made same-sex marriage...",0,"WATCH: Trump Just Told All The Anti-Gay Bigots And Mike Pence To Go F*ck Themselves. A whole lot of evangelical Trump voters just discovered they got duped.By picking Mike Pence as his vice-president, Donald Trump sent a message to the LGBT community that their rights are in jeopardy of being rolled back.In fact, one of the reasons why so many conservative Christians tossed their alleged morality out the window to vote for him was precisely because they believed he would pass a constitutional amendment banning same-sex marriage or pack the courts with anti-gay bigots to strike down all the rulings in favor of marriage equality.But during his interview on 60 minutes with Leslie Stahl on Sunday night, Trump once again backed off another campaign promise. Do you support marriage equality? Stahl asked Trump. It s irrelevant because it s already settled, Trump replied. It s law. It was settled in the Supreme Court. I mean, it s done. When Stahl pointed out that Trump s Supreme Court nom..."
1,"China backs U.N. call for justice in Yemen, U.S. and Saudis don't","GENEVA (Reuters) - China signaled on Wednesday it was willing to back an international inquiry into atrocities in Yemen, as demanded by the U.N. High Commissioner for Human Rights, but Saudi Arabia and the United States said they did not support the idea. For three years running U.N. human rights chief Zeid Ra ad al-Hussein has asked the 47 countries in the U.N. Human Rights Council to set up an independent investigation into Yemen s war, which has killed at least 10,000, destroyed the economy, led to a cholera epidemic and pushed millions to the brink of famine. Despite his pleas, they have twice supported a Saudi plan to let Yemen investigate by itself. On Wednesday, the Netherlands and Canada unveiled a draft resolution to establish an international commission of inquiry (COI) to ensure that perpetrators of violations and abuses, including those that may constitute war crimes and crimes against humanity, are held accountable . The three-page text was supported by many count...",1,"China backs U.N. call for justice in Yemen, U.S. and Saudis don't. GENEVA (Reuters) - China signaled on Wednesday it was willing to back an international inquiry into atrocities in Yemen, as demanded by the U.N. High Commissioner for Human Rights, but Saudi Arabia and the United States said they did not support the idea. For three years running U.N. human rights chief Zeid Ra ad al-Hussein has asked the 47 countries in the U.N. Human Rights Council to set up an independent investigation into Yemen s war, which has killed at least 10,000, destroyed the economy, led to a cholera epidemic and pushed millions to the brink of famine. Despite his pleas, they have twice supported a Saudi plan to let Yemen investigate by itself. On Wednesday, the Netherlands and Canada unveiled a draft resolution to establish an int

In [ ]:
df_model[["title", "text", "content", "label"]].to_csv("../Data/ml/processed_data_en.csv", index=False)